# ETKDG-Orbit E2 代表性输出可视化

直接查看 formal E2 的 `known graph + known Target_PG orbit action → XYZ` 示例。下拉选择 C2/C3 exact、compatible supergroup 或含 Sn 样本；视图可旋转和缩放。

In [1]:
from pathlib import Path
import json
import sys

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'generative_model/runs/etkdg_orbit_e2/point_group_report.json').is_file():
            return candidate
    raise FileNotFoundError('未找到 MARL_for_COFs 项目根目录')

ROOT = find_project_root(Path.cwd())
EXAMPLE_DIR = ROOT / 'generative_model/visualization/e2_examples'
VIS_DIR = ROOT / 'cof_symmetry_pipeline/visualization'
if str(VIS_DIR) not in sys.path:
    sys.path.insert(0, str(VIS_DIR))
from visualize_xyz import build_viewer, parse_xyz

manifest = json.loads((EXAMPLE_DIR / 'examples.json').read_text(encoding='utf-8'))
records = manifest['records']
print(f'项目根目录: {ROOT}')
print(f'E2 示例数量: {len(records)}')

项目根目录: /home/tianyajun/MARL_for_COFs
E2 示例数量: 5


In [3]:
selector = widgets.Dropdown(
    options=[(f"{r['label']} | {r['molecule_id']} | {r['target_pg']}→{r['actual_pg']}", i) for i, r in enumerate(records)],
    description='示例:', layout=widgets.Layout(width='650px')
)
style = widgets.Dropdown(options=['ball-and-stick', 'stick', 'sphere', 'line'], value='ball-and-stick', description='样式:')
labels = widgets.Checkbox(value=False, description='非氢元素标签')
spin = widgets.Checkbox(value=False, description='自动旋转')
button = widgets.Button(description='显示', button_style='primary')
output = widgets.Output()

def render(_=None):
    with output:
        clear_output(wait=True)
        record = records[selector.value]
        metadata = json.loads((EXAMPLE_DIR / record['metadata_file']).read_text(encoding='utf-8'))
        xyz_text, atoms = parse_xyz(EXAMPLE_DIR / record['xyz_file'])
        display(Markdown(
            f"### `{record['molecule_id']}`: Target {record['target_pg']} → Actual {record['actual_pg']}\n"
            f"SMILES: `{metadata['input']['smiles']}`"
        ))
        display(pd.DataFrame([{
            'package_index': record['package_index'],
            'atoms': record['atom_count'],
            'exact': record['pg_exact_match'],
            'compatible': record['pg_compatible'],
            'contains_Sn': record['contains_sn'],
            'selected_seed': metadata['output']['selected_seed'],
            'min_pair_A': metadata['output']['minimum_pair_distance_angstrom'],
            'bond_MAE_A': metadata['output']['bond_length_mae_to_reference_angstrom'],
            'operation_error_A': metadata['output']['maximum_operation_error_angstrom'],
        }]))
        viewer = build_viewer(xyz_text, atoms, style.value, 'white', labels.value, spin.value)
        display(viewer)

button.on_click(render)
display(widgets.HBox([selector, style]), widgets.HBox([labels, spin, button]), output)
render()

Output()

## 如何读取示例

同名 JSON 中的 `input.known_graph` 是 canonical 化学图，`input.known_target_pg_orbit_action` 保存 operation matrices、原子 permutation 和 orbit；XYZ 是 E2 后端的输出。`exact=false, compatible=true` 表示输出具有比目标更高的实际点群，不能当作失败。XYZ 本身不编码键级，Notebook 中显示的键由 3Dmol.js 按距离推断。